# E4 — Independent test of the specificity axis

**Pre-registered in `docs/preregistration/E4.md`, committed before this notebook was written.**
Every threshold, every arm and the reading of the control table come from that file.

C4 built a specificity axis from 154 phage-display profiles and **failed its own rule**
(`validated: false`, median centred r = +0.288, beat the family-average profile on 55 of 115
domains). That failure was measured against the data the model was trained on. This program tests
the axis against **45 dissociation constants measured by a different group, by a different
method** (Kazemein Jasemi et al. 2024, *Cells* 13(2):195).

### The expected outcome is failure, and that is the point

A second, *independent* failure is a stronger negative result than one internal failure. Nothing
in the project rests on this axis: no rewiring claim is made anywhere.

### What the axis actually is (Part 1)

C4 collapses each 14 x 20 PWM to **one information-content-weighted 20-vector**, and the Ridge
head predicts that vector from an ESM-2 embedding of the domain sequence. The axis is therefore a
statement about **amino-acid composition, not position**: two peptides with the same composition
in a different order are indistinguishable to it. Peptides are scored accordingly — mean
predicted log-preference per residue, no window, no alignment.

### The leakage problem, and what is done about it (Part 4)

| | domains | measurements | prediction used |
|---|---|---|---|
| in C4's training set | 12 | 37 | **leave-cluster-out**, from a re-run of C4's own CV |
| never in it | 4 | 7 | the final model |
| excluded — no domain sequence | 1 (ARHGEF30) | 1 | — |

### The control that decides what a failure means (Part 6)

Everything is repeated with the domain's **own observed phage-display profile**. If the observed
profile fails too, the honest claim is about the data, not about C4. That reading was fixed in
advance.

Runtime: about 10 minutes. A GPU makes the embedding step faster but is not required.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, re
from pathlib import Path
import pandas as pd, numpy as np
from scipy.stats import spearmanr, binomtest, t as tdist

DRIVE = Path('/content/drive/MyDrive')

def pick(*names, what=''):
    for n in names:
        if (DRIVE / n).exists(): return DRIVE / n
    raise FileNotFoundError(f'No {what} folder in MyDrive. Tried: ' + ', '.join(names))

PROJECT = pick('sh3_repro/extended', 'SH3_extended', 'SH3 extended', what='project')
FROZEN  = pick('sh3_repro', what='frozen-run')          # READ ONLY
FROZEN_IN = PROJECT / 'input' / 'frozen'
BIND    = PROJECT / 'input' / 'external' / 'binding'
OUT     = PROJECT / 'results' / 'E4_specificity_independent'
OUT.mkdir(parents=True, exist_ok=True)

def frozen(rel, bundled):
    p = FROZEN / 'results' / rel
    return p if p.exists() else FROZEN_IN / 'results' / bundled

ALPHA, AAS = 0.025, list('ACDEFGHIKLMNPQRSTVWY')
print('project:', PROJECT); print('frozen :', FROZEN)

## 1. The pinned inputs

C4's collapsed profiles are read straight from its recorded output, so the profiles E4 tests are
byte-identical to the ones C4 reported — the PWM archive is not re-parsed.

In [ ]:
kd  = pd.read_csv(BIND / 'kazemein_jasemi_2024_TableS6_Kd.csv')
pep = pd.read_csv(BIND / 'kazemein_jasemi_2024_TableS2_peptides.csv')
PR  = pd.read_csv(frozen('C4_specificity_axis/pwm_profiles.csv', 'pwm_profiles.csv'))

assert len(kd) == 45 and kd.sh3_domain.nunique() == 17, 'Kd table is not the pinned one'
assert len(pep) == 12 and len(PR) == 154, 'peptide or profile table is not the pinned one'
assert abs(kd.Kd_uM.min() - 0.2) < 1e-9 and abs(kd.Kd_uM.max() - 125.0) < 1e-9, \
       'Kd range does not match the paper (0.2-125 uM)'

# each peptide sequence must match its published residue coordinates
assert (pep.end - pep.start + 1 == pep.sequence.str.len()).all()
assert (pep.length == pep.sequence.str.len()).all()

kd['logKd'] = np.log10(kd.Kd_uM)
print(f'{len(kd)} affinities, {kd.sh3_domain.nunique()} domains, '
      f'{kd.peptide.nunique()} of {len(pep)} peptides measured')
print(f'{len(PR)} C4 profiles over {PR.domain.nunique()} domain sequences')

## 2. Rebuild C4's domain table, exactly as C4 did

`DOM` averages the profiles of domains carrying more than one PWM — C4's own rule. `Y` is the
centred-log-ratio transform it regressed on.

In [ ]:
DOM = PR.groupby(['domain', 'seq'], as_index=False)[AAS].mean()
assert len(DOM) == 115, f'expected C4 115 domains, got {len(DOM)}'

clr  = lambda P: (np.log(np.clip(P, 1e-6, None))
                  - np.log(np.clip(P, 1e-6, None)).mean(1, keepdims=True))
Y = clr(DOM[AAS].values)
print(f'{len(DOM)} domains, profile matrix {Y.shape}')

# the four domains C4 never saw: sequences come from the atlas
EXTRA = {'ABI1': ('ABI1', 1), 'DLG2': ('DLG2', 1), 'NCK1-3': ('NCK1', 3),
         'SORBS1-1': ('SORBS1', 1)}
atlas = pd.read_csv(frozen('A0_cancer_atlas/SH3_cancer_master.tsv', 'SH3_cancer_master.tsv'),
                    sep='\t', usecols=['gene', 'domain_idx', 'wt_domain_seq'])
aseq = {(g, int(i)): s for g, i, s in atlas.drop_duplicates(['gene', 'domain_idx']).itertuples(index=False)}
extra_seq = {k: aseq[v] for k, v in EXTRA.items()}
for k, s in extra_seq.items(): print(f'  {k:10s} {len(s)} residues')

## 3. ESM-2 embeddings and C4's cross-validation, re-run

The same representation C4 and A2 use: last hidden layer, mean over residues. The CV repeats C4's
clustering (single linkage at >= 40 % identity) and its per-fold alpha selection, so the
leave-cluster-out predictions E4 scores are the ones C4's own verdict was computed from.

**This cell verifies the re-run against C4's recorded decision.** If the median centred r does not
come back at +0.288 and `validated: false`, the re-run is not C4 and nothing below it is valid.

In [ ]:
os.system('pip -q install transformers biopython scikit-learn')
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from Bio import Align
from sklearn.linear_model import Ridge

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained('facebook/esm2_t33_650M_UR50D')
esm = AutoModelForMaskedLM.from_pretrained('facebook/esm2_t33_650M_UR50D').to(DEV).eval()
if DEV == 'cuda': esm = esm.half()
print('device:', DEV)

@torch.no_grad()
def mean_embed(seq):
    enc = tok(seq, return_tensors='pt').to(DEV)
    h = esm(**enc, output_hidden_states=True).hidden_states[-1]
    return h[0, 1:len(seq) + 1].float().mean(0).cpu().numpy()

XP = np.stack([mean_embed(s) for s in DOM.seq])
XE = np.stack([mean_embed(extra_seq[k]) for k in EXTRA])
print('embeddings:', XP.shape, XE.shape)

In [ ]:
# C4's clustering: single linkage at >= 40% identity, gap-penalised pairwise alignment
idal = Align.PairwiseAligner(open_gap_score=-11, extend_gap_score=-1)
idal.match_score, idal.mismatch_score = 1, 0
n = len(DOM); ID = np.eye(n)
for i in range(n):
    for j in range(i + 1, n):
        ID[i, j] = ID[j, i] = idal.score(DOM.seq[i], DOM.seq[j]) / min(len(DOM.seq[i]), len(DOM.seq[j]))
lab = np.arange(n)
for i in range(n):
    for j in range(n):
        if ID[i, j] >= 0.40: lab[lab == lab[j]] = lab[i]
print(f'{n} domains in {len(np.unique(lab))} clusters')

# leave-one-cluster-out, with C4's per-fold alpha selection
OOF   = np.zeros_like(Y)          # out-of-fold predicted profile
YBAR  = np.zeros_like(Y)          # the average profile that fold saw
alphas_used, r_model = [], np.zeros(n)
for c in np.unique(lab):
    te, tr = np.where(lab == c)[0], np.where(lab != c)[0]
    mu, sd = XP[tr].mean(0), XP[tr].std(0) + 1e-6; ybar = Y[tr].mean(0)
    a = max([1, 10, 100, 1000], key=lambda a: np.mean(
        [np.corrcoef(Ridge(alpha=a).fit((XP[np.setdiff1d(tr, [k])] - mu) / sd,
                                       Y[np.setdiff1d(tr, [k])])
                     .predict(((XP[k] - mu) / sd)[None])[0] - ybar, Y[k] - ybar)[0, 1]
         for k in tr[::4]]))
    pred = Ridge(alpha=a).fit((XP[tr] - mu) / sd, Y[tr]).predict((XP[te] - mu) / sd)
    for k, tt in enumerate(te):
        OOF[tt], YBAR[tt] = pred[k], ybar
        r_model[tt] = np.corrcoef(pred[k] - ybar, Y[tt] - ybar)[0, 1]
    alphas_used.append(a)

R_MED = float(np.median(r_model))
rec = json.loads(Path(frozen('C4_specificity_axis/specificity_decision.json',
                            'specificity_decision.json')).read_text())
print(f'median centred r  this re-run {R_MED:+.3f}   C4 recorded {rec["median_centred_r"]:+.3f}')
assert abs(R_MED - rec['median_centred_r']) < 0.03, 'the re-run is not C4'
assert rec['validated'] is False
print('re-run reproduces C4, which returned validated: false')

# the final model, for the four domains C4 never saw
mu, sd = XP.mean(0), XP.std(0) + 1e-6
a_fin = int(pd.Series(alphas_used).mode().iat[0])
final = Ridge(alpha=a_fin).fit((XP - mu) / sd, Y)
PRED_EXTRA = final.predict((XE - mu) / sd)
YBAR_ALL = Y.mean(0)
print(f'final model alpha {a_fin}')

## 4. Map the measured domains onto C4's entries (Part 4)

The paper names domains `GENE` or `GENE-i`; C4 names them `GENE-i/n`, with `-ISk/m` suffixes for
isoform-specific variants. Where several entries match, their profiles are averaged — C4's rule
for domains with two PWMs.

In [ ]:
def c4_entries(name):
    """paper domain name -> indices of the matching C4 domain entries"""
    base, idx = (name.rsplit('-', 1) if re.search(r'-\d+$', name) else (name, '1'))
    pat = re.compile(rf'^{re.escape(base)}-{idx}/')
    return [i for i, d in enumerate(DOM.domain) if pat.match(d)]

MAP, rows = {}, []
for name in sorted(kd.sh3_domain.unique()):
    hits = c4_entries(name)
    if hits:      MAP[name] = ('trained', hits)
    elif name in EXTRA: MAP[name] = ('held out', [list(EXTRA).index(name)])
    else:         MAP[name] = ('excluded', [])
    rows.append((name, MAP[name][0], ', '.join(DOM.domain[i] for i in hits) or '-',
                 int((kd.sh3_domain == name).sum())))
M = pd.DataFrame(rows, columns=['measured_as', 'group', 'C4_entries', 'n_Kd'])
print(M.to_string(index=False))

n_tr = M[M.group == 'trained'].n_Kd.sum(); n_ho = M[M.group == 'held out'].n_Kd.sum()
print(f'\ntrained {(M.group=="trained").sum()} domains / {n_tr} measurements | '
      f'held out {(M.group=="held out").sum()} / {n_ho} | '
      f'excluded {(M.group=="excluded").sum()} / {M[M.group=="excluded"].n_Kd.sum()}')
assert n_tr == 37 and n_ho == 7, 'the split is not the pre-registered 37 / 7'

## 5. Score every measured pair three ways (Part 3)

`S(d, p) = sum_a f_p(a) * profile_d(a)` — the mean log-preference per residue, in the same
centred-log-ratio space the model was fitted in. Three profiles: the **model** prediction under
test, the domain-agnostic **average**, and the domain's **observed** phage-display profile.

In [ ]:
PSEQ = dict(zip(pep.peptide, pep.sequence))
def comp(s):
    v = np.array([s.count(a) for a in AAS], float)
    return v / v.sum()
COMP = {p: comp(s) for p, s in PSEQ.items()}

recs = []
for r in kd.itertuples():
    grp, hits = MAP[r.sh3_domain]
    if grp == 'excluded': continue
    f = COMP[r.peptide]
    if grp == 'trained':
        y_model = OOF[hits].mean(0); y_avg = YBAR[hits].mean(0); y_obs = Y[hits].mean(0)
    else:
        y_model = PRED_EXTRA[hits[0]]; y_avg = YBAR_ALL; y_obs = None
    recs.append(dict(domain=r.sh3_domain, peptide=r.peptide, group=grp, Kd_uM=r.Kd_uM,
                     logKd=r.logKd, S_model=float(f @ y_model), S_avg=float(f @ y_avg),
                     S_obs=float(f @ y_obs) if y_obs is not None else np.nan))
D = pd.DataFrame(recs)
D.to_csv(OUT / 'scored_pairs_E4.csv', index=False)
assert len(D) == 44, f'expected the pre-registered 44 measurements, got {len(D)}'
print(f'{len(D)} scored pairs, {D.domain.nunique()} domains')
print(D.head(8).to_string(index=False))

## 6. Experiment 1 — MAIN

Within-domain-centred Pearson r over the 12 trained-on domains, `df = n - k - 1`. Every
domain-level effect is removed, so what is left is specificity: which peptide *this* domain
prefers.

**A verdict of yes needs r < 0 at p < 0.025 *and* |r| above the same statistic for the average
profile.** At df = 24 that means |r| >= 0.44; a weaker real signal returns a null, as stated in
advance.

In [ ]:
def rmcorr(d, col, ycol='logKd'):
    """Pearson r on within-domain-centred values; df = n - k - 1."""
    # a domain measured once carries no within-domain information: centring sends it to 0,
    # so it would spend a degree of freedom for nothing. Matters only to Exp 5a.
    d = d[d.groupby('domain').domain.transform('size') >= 2]
    g = d.groupby('domain')
    x = (d[col] - g[col].transform('mean')).values
    y = (d[ycol] - g[ycol].transform('mean')).values
    n, k = len(d), d.domain.nunique()
    df = n - k - 1
    if df < 1 or np.std(x) < 1e-12: return dict(r=np.nan, p=np.nan, n=n, k=k, df=df)
    r = float(np.corrcoef(x, y)[0, 1])
    p = float(2 * tdist.sf(abs(r) * np.sqrt(df / max(1 - r * r, 1e-12)), df))
    return dict(r=r, p=p, n=n, k=k, df=df)

TR = D[D.group == 'trained']
e1_model = rmcorr(TR, 'S_model'); e1_avg = rmcorr(TR, 'S_avg')
e1_obs   = rmcorr(TR, 'S_obs')

beats = abs(e1_model['r']) > abs(e1_avg['r'])
E1_VERDICT = ('yes' if (e1_model['r'] < 0 and e1_model['p'] < ALPHA and beats) else 'no')
for lab, v in [('model', e1_model), ('average profile', e1_avg), ('observed profile', e1_obs)]:
    print(f'{lab:18s} r {v["r"]:+.3f}  p {v["p"]:.4g}  (n {v["n"]}, k {v["k"]}, df {v["df"]})')
print(f'\nbeats the average profile: {beats}     MAIN VERDICT: {E1_VERDICT}')

## 7. Experiment 2 — the assumption-free version

Per-domain Spearman for domains with at least 3 measurements, aggregated by a two-sided sign test
on the direction. Declared in advance: with 8 domains this reaches 0.025 **only at 8/8**.

In [ ]:
def per_domain(d, col):
    rows = []
    for dom, g in d.groupby('domain'):
        if len(g) < 3: continue
        rho, p = spearmanr(g[col], g.logKd)
        rows.append(dict(domain=dom, n=len(g), rho=float(rho), p=float(p)))
    return pd.DataFrame(rows)

def sign_test(t):
    if not len(t): return dict(k=0, n=0, p=np.nan, median=np.nan, lo=np.nan, hi=np.nan)
    k = int((t.rho < 0).sum())
    bs = [np.median(np.random.default_rng(b).choice(t.rho.values, len(t))) for b in range(2000)]
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return dict(k=k, n=len(t), p=float(binomtest(k, len(t), 0.5).pvalue),
                median=float(t.rho.median()), lo=float(lo), hi=float(hi))

PD_model, PD_obs = per_domain(TR, 'S_model'), per_domain(TR, 'S_obs')
PD_model.to_csv(OUT / 'per_domain_E4.csv', index=False)
s_model, s_obs = sign_test(PD_model), sign_test(PD_obs)
print(PD_model.to_string(index=False))
MIN_DOM = 8
E2_VERDICT = ('too few' if s_model['n'] < MIN_DOM else
              'yes' if (s_model['p'] < ALPHA and s_model['hi'] < 0) else 'no')
for lab, s in [('model', s_model), ('observed', s_obs)]:
    print(f'{lab:9s} {s["k"]}/{s["n"]} negative, sign test p {s["p"]:.4g}, '
          f'median rho {s["median"]:+.3f} [{s["lo"]:+.3f}, {s["hi"]:+.3f}]')
print(f'\nEXP 2 VERDICT: {E2_VERDICT}')

## 8. Experiments 3, 4 and 5 — reported, no verdict

Exp 3 pools every measurement with no domain centring: **peptide identity dominates it**, so it is
reported and deliberately carries no verdict. Exp 4 is the four domains C4 never saw. Exp 5 is the
two sensitivity checks.

In [ ]:
rho3, p3 = spearmanr(D.S_model, D.logKd)
print(f'Exp 3  pooled, no centring: rho {rho3:+.3f}  p {p3:.4g}  (n {len(D)})  - NO VERDICT')

HO = D[D.group == 'held out']
rho4, p4 = spearmanr(HO.S_model, HO.logKd) if len(HO) > 2 else (np.nan, np.nan)
print(f'Exp 4  never trained on: {HO.domain.nunique()} domains, {len(HO)} measurements, '
      f'rho {rho4:+.3f} - NO VERDICT at any n')

e5a = rmcorr(TR[TR.peptide != 'RP1'], 'S_model')
TRr = TR.copy()
for c in ('S_model', 'logKd'):
    TRr[c] = TRr.groupby('domain')[c].rank()
e5b = rmcorr(TRr, 'S_model')
print(f'Exp 5a dropping RP1 (nested in P3): r {e5a["r"]:+.3f}  p {e5a["p"]:.4g}  (n {e5a["n"]})')
print(f'Exp 5b within-domain ranks:        r {e5b["r"]:+.3f}  p {e5b["p"]:.4g}')

## 9. The control (Part 6) — does the *observed* profile do any better?

The reading was fixed before any of these numbers existed:

| observed | model | conclusion |
|---|---|---|
| fails | fails | the measurement, not the model — E4 cannot convict C4 specifically |
| passes | fails | the model — the approach works and C4's prediction of it does not |
| — | passes | post-hoc success; C4's rule already returned false |

In [ ]:
obs_pass = bool(e1_obs['r'] < 0 and e1_obs['p'] < ALPHA)
mod_pass = E1_VERDICT == 'yes'
CONCLUSION = ('post-hoc success - C4 rule already returned false' if mod_pass else
              'the model: the observed profile predicts affinity and the prediction of it does not'
              if obs_pass else
              'the measurement, not the model: compositional profiles do not predict these affinities')
print(f'observed profile passes: {obs_pass}   model passes: {mod_pass}')
print('CONCLUSION (pre-registered reading):', CONCLUSION)

## 10. RESULTS_E4.md and metrics_E4.json

In [ ]:
M.to_csv(OUT / 'domain_mapping_E4.csv', index=False)
met = dict(program='E4', date=pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'),
           preregistration='docs/preregistration/E4.md', alpha=ALPHA,
           n_measurements=int(len(D)), n_domains=int(D.domain.nunique()),
           n_excluded=int(45 - len(D)),
           E1_main=dict(**e1_model, verdict=E1_VERDICT, beats_average=bool(beats)),
           E1_average=e1_avg, E1_observed=e1_obs,
           E2=dict(**s_model, verdict=E2_VERDICT), E2_observed=s_obs,
           E3_pooled=dict(rho=float(rho3), p=float(p3), n=int(len(D)), verdict='none by design'),
           E4_heldout=dict(rho=float(rho4) if rho4 == rho4 else None,
                           n=int(len(HO)), domains=int(HO.domain.nunique()),
                           verdict='none by design'),
           E5a_drop_RP1=e5a, E5b_ranks=e5b,
           observed_profile_passes=obs_pass, conclusion=CONCLUSION,
           c4_recorded_median_centred_r=rec['median_centred_r'],
           c4_validated=rec['validated'], rerun_median_centred_r=R_MED)
json.dump(met, open(OUT / 'metrics_E4.json', 'w'), indent=1)

f = lambda v: '-' if v != v else f'{v:+.3f}'
L = ['# E4 - independent test of the specificity axis', '',
     f'Run {met["date"]}. Machine-written. Plan fixed in advance: `docs/preregistration/E4.md`.', '',
     'C4 built this axis and failed its own pre-registered rule (`validated: false`, median '
     f'centred r {rec["median_centred_r"]:+.3f}). The cross-validation is re-run here and '
     f'reproduces it at {R_MED:+.3f}, so the predictions scored below are the ones that verdict '
     'came from.', '',
     f'{len(D)} of 45 measurements over {D.domain.nunique()} of 17 domains; ARHGEF30 excluded '
     'for want of a domain sequence. The axis is compositional, so peptides are scored by '
     'amino-acid frequency.', '',
     '## Experiment 1 - MAIN: within-domain-centred correlation', '',
     '| profile | r | p | verdict |', '|---|---|---|---|',
     f'| **model** (leave-cluster-out) | {f(e1_model["r"])} | {e1_model["p"]:.3g} | **{E1_VERDICT}** |',
     f'| average profile (domain-agnostic) | {f(e1_avg["r"])} | {e1_avg["p"]:.3g} | baseline |',
     f'| observed phage-display profile | {f(e1_obs["r"])} | {e1_obs["p"]:.3g} | control |', '',
     f'n = {e1_model["n"]} over k = {e1_model["k"]} domains, df = {e1_model["df"]}. Negative r is '
     'the direction predicted in advance. A verdict of yes needed p < 0.025 and a larger |r| than '
     f'the average profile; beats the average: {beats}.', '',
     '## Experiment 2 - per-domain sign test', '',
     f'| model | {s_model["k"]}/{s_model["n"]} in the predicted direction | sign test p '
     f'{s_model["p"]:.3g} | median rho {f(s_model["median"])} [{f(s_model["lo"])}, {f(s_model["hi"])}] | **{E2_VERDICT}** |',
     f'| observed | {s_obs["k"]}/{s_obs["n"]} | p {s_obs["p"]:.3g} | median rho {f(s_obs["median"])} | control |', '',
     '## Reported without a verdict', '',
     f'- **Exp 3**, pooled with no domain centring: rho {f(rho3)} (p {p3:.3g}, n {len(D)}). Peptide '
     'identity dominates this number, which is why it was given no verdict in advance.',
     f'- **Exp 4**, the {HO.domain.nunique()} domains C4 never saw: {len(HO)} measurements, '
     f'rho {f(rho4)}. Too few for any verdict, declared in advance.',
     f'- **Exp 5a**, dropping RP1 (nested in P3): r {f(e5a["r"])} (p {e5a["p"]:.3g}).',
     f'- **Exp 5b**, within-domain ranks: r {f(e5b["r"])} (p {e5b["p"]:.3g}).', '',
     '## Reading, by the rule fixed in advance', '',
     f'**{CONCLUSION}**', '',
     'Power, stated before the result existed: df = 24 detects only |r| >= 0.44, and the '
     '8-domain sign test reaches 0.025 only at 8/8. A null here means no effect large enough for '
     f'{len(D)} measurements to see - not no effect.']
(OUT / 'RESULTS_E4.md').write_text('\n'.join(L))
print('\n'.join(L))

In [ ]:
print('written to', OUT)
for p in sorted(OUT.iterdir()): print(f'  {p.stat().st_size:>9,} B  {p.name}')